Requirements:\
datasets.py - Ensure datasets file points to correct directory\
training.py\
unetFixed.py\
SplitNetInterp.py\
pytorch\
numpy\
tqdm\
matplotlib\
\
For correct test simulations, also use test_sims.npy

In [1]:
import datasets
from training import darcy_loss
from torch.optim import Optimizer
import torch
from torch.utils.data import DataLoader
from torch import nn
from torch.optim import Adam
import numpy as np
from tqdm import tqdm
from unetFixed import UNet,UNetShort
import SplitNetInterp

import matplotlib.pyplot as plt

torch.random.manual_seed(42)

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def just_darcy(out) -> torch.Tensor:

    # If we assume the output is in order k,pres,phi
    # pres_grad is the gradient of the pressure along the y and x directions as a tuple
    pres_grad = torch.gradient(out[:, 1:2], dim=(-2,-1))

    # get velocity by multiplying the gradient by the conductivity
    y_grad = pres_grad[0] * out[:, 0:1]
    x_grad = pres_grad[1] * out[:, 0:1]

    # compute the divergence by the second derivative of the gradients and adding them together
    yy_grad = torch.gradient(y_grad, spacing=(1,),dim=(-2,))[0]
    xx_grad = torch.gradient(x_grad, spacing=(1,),dim=(-1,))[0]
    final = yy_grad + xx_grad

    # total divergence should be 0
    loss = (final**2)

    return loss

In [2]:
# Dataset
using_default_split = True
n_sims = 500

if using_default_split:
    test_sims = np.load("../test_sims.npy")
    test_sims = test_sims[test_sims < n_sims]
else:
    test_sims = np.arange(0,n_sims)

test_data = datasets.BorderDenseDatasetFullSides(test_sims, channels="KP" )
test_loader = torch.utils.data.DataLoader(test_data, batch_size=8)

crit = nn.MSELoss()

In [3]:
# Load model
model_path = "minimum_info\Modelb_3x3lines_showsides_SplitInterp256.pt"
model = torch.load(model_path, weights_only=False).to(device)

<>:2: SyntaxWarning: invalid escape sequence '\M'
<>:2: SyntaxWarning: invalid escape sequence '\M'
/var/folders/qm/z45q_dj168lbjph6mvbqnhg40000gn/T/ipykernel_38095/3246511943.py:2: SyntaxWarning: invalid escape sequence '\M'
  model_path = "minimum_info\Modelb_3x3lines_showsides_SplitInterp256.pt"
/var/folders/qm/z45q_dj168lbjph6mvbqnhg40000gn/T/ipykernel_38095/3246511943.py:2: SyntaxWarning: invalid escape sequence '\M'
  model_path = "minimum_info\Modelb_3x3lines_showsides_SplitInterp256.pt"


FileNotFoundError: [Errno 2] No such file or directory: 'minimum_info\\Modelb_3x3lines_showsides_SplitInterp256.pt'

In [4]:
# Pick sample to test
simulation_index = 1 # 0 to 118 (with 500 total samples)
simulation_step = 0 # 0 to 199

model_title = "BC+Sides: SplitNet with upscale"

model.eval()
with torch.inference_mode():
    feat,label = test_data[simulation_index * 200 + simulation_step]
    feat = feat.cuda().unsqueeze(0)
    label = label.cuda().unsqueeze(0)
    p_loss, out = darcy_loss(model, feat)
model.train()
channel = 1

plt.imshow(out.detach().cpu().numpy()[0,channel], vmin=-1,vmax=1)
plt.title(f"{model_title}: output {"pressure" if channel == 1 else "conductivity"}")
plt.colorbar()
plt.xticks([])
plt.yticks([])
plt.show()

plt.imshow(label.detach().cpu().numpy()[0,channel], vmin=-1,vmax=1)
plt.title(f"{model_title}: Real {"pressure" if channel == 1 else "conductivity"}")
plt.colorbar()
plt.xticks([])
plt.yticks([])
plt.show()

plt.imshow(feat.detach().cpu().numpy()[0,channel], vmin=-1,vmax=1)
plt.title(f"{model_title}: input {"pressure" if channel == 1 else "conductivity"}")
plt.colorbar()
plt.xticks([])
plt.yticks([])
plt.show()

plt.imshow(just_darcy(out).detach().cpu().numpy()[0,0])
plt.title(f"{model_title}: output darcy loss map {"pressure" if channel == 1 else "conductivity"}")
plt.colorbar()
plt.xticks([])
plt.yticks([])
plt.show()

NameError: name 'model' is not defined

In [5]:
# Compute mean loss across the entire dataset
test_loss = 0
test_darcy = 0
pressure_loss = 0
conductivity_loss = 0
with torch.no_grad(), torch.inference_mode(True):
    for feat,label in tqdm(test_loader):

        feat = feat.to(device)
        label = label.to(device)
        p_loss, out = darcy_loss(model, feat)
        test_darcy += p_loss.item()
        cond_loss = crit(out[:,[0]], label[:,[0]]) * 0.5
        pres_loss = crit(out[:,[1]], label[:,[1]]) * 0.5
        loss = cond_loss + pres_loss + p_loss
        test_loss += loss.item()
        conductivity_loss += cond_loss.item()
        pressure_loss += pres_loss.item()

test_loss /= test_loader.__len__()
test_darcy /= test_loader.__len__()
conductivity_loss /= test_loader.__len__()
pressure_loss /= test_loader.__len__()

print(f"Mean Loss: {test_loss}")
print(f"Darcy Loss: {test_darcy}")
print(f"Conductivity Loss: {conductivity_loss}")
print(f"Pressure Loss: {pressure_loss}")

  0%|          | 0/2950 [00:00<?, ?it/s]


NameError: name 'model' is not defined